# Hyperspectral Classification on Indian Pines
**kNN → classical ML → PCA → spectral-spatial → deep learning (1D-CNN, HybridSN)**

This notebook loads the 200-band image cube from `indian_pines.csv`, reconstructs its
145 × 145 spatial layout using the `row` and `col` columns, and loads the separate
ground-truth labels for supervised classification.

Pipeline:
1. Load the CSV image cube and ground-truth labels
2. Shared stratified train/test split
3. Spectral-only classifiers: kNN, Logistic Regression, LDA, Random Forest, SVM
4. PCA dimensionality reduction
5. Spectral-spatial classifiers using neighbourhood patches
6. Deep learning: 1D-CNN and HybridSN (PyTorch)
7. Results, confusion matrix, classification maps, and multi-seed evaluation

**Required files in the notebook's working directory:**
- `indian_pines.csv` — contains `row`, `col`, and `band_1` … `band_200`
- One ground-truth file:
  - `Indian_pines_gt.mat`, or
  - `indian_pines_gt.csv` / `indian_pines_labels.csv`, or
  - `Indian_pines_gt.npy`

The spectral CSV does not contain class labels. A separate ground-truth file is
required for supervised classification; the notebook will stop with a clear error
if it is missing rather than inventing labels.

## 1. Setup and data loading

The spectral cube is reconstructed from `indian_pines.csv`. Ground-truth labels
must be supplied separately because the CSV shown for this project contains only
`row`, `col`, and 200 spectral-band columns.

In [4]:
import os, time, warnings
import numpy as np
import pandas as pd
import scipy.io as sio
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap

from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import (accuracy_score, recall_score, cohen_kappa_score,
                             confusion_matrix, classification_report)

warnings.filterwarnings("ignore")
SEED = 42
np.random.seed(SEED)
plt.rcParams["figure.dpi"] = 110

In [5]:
# ---------- Load image cube from CSV ----------
CSV_FILE = "indian_pines.csv"

df = pd.read_csv(CSV_FILE)

band_cols = sorted(
    [c for c in df.columns if c.startswith("band_")],
    key=lambda x: int(x.split("_")[1])
)

required = {"row", "col"}
if not required.issubset(df.columns):
    raise ValueError("CSV must contain 'row' and 'col' columns.")

if len(band_cols) != 200:
    raise ValueError(f"Expected 200 spectral bands, found {len(band_cols)}.")

df["row"] = df["row"].astype(int)
df["col"] = df["col"].astype(int)

H, W, B = 145, 145, len(band_cols)

if len(df) != H * W:
    raise ValueError(f"Expected {H*W} pixels, found {len(df)}.")
if df.duplicated(["row", "col"]).any():
    raise ValueError("Duplicate row/col coordinates found in CSV.")
if not df["row"].between(0, H - 1).all() or not df["col"].between(0, W - 1).all():
    raise ValueError("CSV coordinates must be zero-based and lie in [0, 144].")

cube = np.empty((H, W, B), dtype=np.float32)
cube[df["row"].to_numpy(), df["col"].to_numpy(), :] = (
    df[band_cols].to_numpy(dtype=np.float32)
)
X_all = cube.reshape(H * W, B)

# ---------- Load ground-truth labels ----------
# Supported formats:
# 1) Indian_pines_gt.mat containing variable 'indian_pines_gt'
# 2) Indian_pines_gt.npy: a (145,145) label matrix
# 3) indian_pines_gt.csv / indian_pines_labels.csv:
#    either a 145x145 matrix, or columns row,col,label (or gt/class)

gt = None
gt_mat = "Indian_pines_gt.mat"
gt_npy = "Indian_pines_gt.npy"
gt_csv_candidates = ["indian_pines_gt.csv", "indian_pines_labels.csv"]

if os.path.exists(gt_mat):
    gt = sio.loadmat(gt_mat)["indian_pines_gt"]
elif os.path.exists(gt_npy):
    gt = np.load(gt_npy)
else:
    for gt_file in gt_csv_candidates:
        if os.path.exists(gt_file):
            gdf = pd.read_csv(gt_file)
            label_col = next(
                (c for c in ["label", "gt", "class", "ground_truth"] if c in gdf.columns),
                None
            )
            if {"row", "col"}.issubset(gdf.columns) and label_col is not None:
                gt = np.zeros((H, W), dtype=np.int64)
                gt[
                    gdf["row"].astype(int).to_numpy(),
                    gdf["col"].astype(int).to_numpy()
                ] = gdf[label_col].astype(int).to_numpy()
            elif gdf.shape == (H, W):
                gt = gdf.to_numpy()
            else:
                raise ValueError(
                    f"{gt_file} must be a 145x145 label matrix or contain "
                    "row, col, and one of label/gt/class/ground_truth."
                )
            break

if gt is None:
    raise FileNotFoundError(
        "Ground-truth labels are missing. Your indian_pines.csv contains only "
        "coordinates and spectral bands, not class labels. Add "
        "Indian_pines_gt.mat, Indian_pines_gt.npy, indian_pines_gt.csv, or "
        "indian_pines_labels.csv to the notebook directory."
    )

gt = np.asarray(gt).squeeze().astype(np.int64)
if gt.shape != (H, W):
    raise ValueError(f"Ground truth must have shape {(H, W)}, got {gt.shape}.")
if not np.isin(np.unique(gt), np.arange(17)).all():
    raise ValueError("Ground-truth labels must be integers from 0 to 16.")

N = H * W
y_all = gt.reshape(N)

print("Image cube:", cube.shape)
print("Flattened spectra:", X_all.shape)
print("Ground truth:", gt.shape)
print("Classes:", np.unique(gt))

CLASS_NAMES = ["Alfalfa", "Corn-notill", "Corn-mintill", "Corn", "Grass-pasture",
               "Grass-trees", "Grass-pasture-mowed", "Hay-windrowed", "Oats",
               "Soybean-notill", "Soybean-mintill", "Soybean-clean", "Wheat", "Woods",
               "Bldg-Grass-Tree-Drives", "Stone-Steel-Towers"]
CLASSES = np.arange(1, 17)
NC = len(CLASSES)

FileNotFoundError: Ground-truth labels are missing. Your indian_pines.csv contains only coordinates and spectral bands, not class labels. Add Indian_pines_gt.mat, Indian_pines_gt.npy, indian_pines_gt.csv, or indian_pines_labels.csv to the notebook directory.

### Quick look at the data

In [ ]:
cmap = ListedColormap(["black"] + [plt.cm.tab20(i) for i in range(16)])

def false_colour(c, bands=(50, 27, 17)):
    rgb = c[:, :, list(bands)].copy()
    for i in range(3):
        lo, hi = np.percentile(rgb[:, :, i], (2, 98))
        rgb[:, :, i] = np.clip((rgb[:, :, i] - lo) / (hi - lo + 1e-9), 0, 1)
    return rgb

fig, ax = plt.subplots(1, 2, figsize=(10, 5))
ax[0].imshow(false_colour(cube)); ax[0].set_title("False-colour composite")
ax[1].imshow(gt, cmap=cmap, vmin=0, vmax=16, interpolation="nearest"); ax[1].set_title("Ground truth")
for a in ax: a.axis("off")
plt.tight_layout(); plt.show()

counts = pd.Series(y_all[y_all > 0]).value_counts().sort_index()
print(f"Labeled pixels: {counts.sum()} of {N}")
pd.DataFrame({"class": CLASS_NAMES, "pixels": counts.values}, index=CLASSES)

In [ ]:
# Mean spectrum of every class
plt.figure(figsize=(9, 4.5))
for c in CLASSES:
    plt.plot(X_all[y_all == c].mean(0), label=CLASS_NAMES[c - 1], color=plt.cm.tab20(c - 1), lw=1.2)
plt.xlabel("Band index"); plt.ylabel("Mean reflectance (DN)")
plt.title("Mean spectral signature per class")
plt.legend(fontsize=6, ncol=2); plt.tight_layout(); plt.show()

## 2. Train / test split (shared by all models)

A stratified split of the **labeled** pixels. `TRAIN_FRAC` is the share used for training; change it to
e.g. 0.10 to reproduce the common "10 % training" protocol.

In [ ]:
TRAIN_FRAC = 0.20

labeled_idx = np.where(y_all > 0)[0]
class_counts = pd.Series(y_all[labeled_idx]).value_counts()
if (class_counts < 2).any():
    raise ValueError("Every class needs at least 2 labeled pixels for stratified train/test splitting.")

tr_idx, te_idx = train_test_split(
    labeled_idx, train_size=TRAIN_FRAC,
    stratify=y_all[labeled_idx], random_state=SEED
)
print(f"train: {len(tr_idx)} | test: {len(te_idx)}")

# Standardise bands using TRAIN statistics only (no leakage)
scaler = StandardScaler().fit(X_all[tr_idx])
X_spec = scaler.transform(X_all).astype(np.float32)
y_tr = y_all[tr_idx]

## 3. Evaluation helpers

In [ ]:
results, per_class_acc, pred_all = {}, {}, {}

def evaluate(name, pred_full, secs=None):
    """pred_full = predictions for ALL N pixels; metrics are computed on the test pixels."""
    yt, yp = y_all[te_idx], pred_full[te_idx]
    per_class = recall_score(yt, yp, labels=CLASSES, average=None, zero_division=0)
    results[name] = dict(OA=100 * accuracy_score(yt, yp), AA=100 * per_class.mean(),
                         Kappa=cohen_kappa_score(yt, yp), Time_s=secs)
    per_class_acc[name] = 100 * per_class
    pred_all[name] = pred_full
    r = results[name]
    print(f"{name:<32} OA={r['OA']:.2f}  AA={r['AA']:.2f}  Kappa={r['Kappa']:.4f}"
          + (f"  ({secs:.1f}s)" if secs else ""))

def tune_and_run(name, estimator, grid, X):
    """3-fold CV grid search on the TRAINING pixels, then predict every pixel."""
    t = time.time()
    gs = GridSearchCV(estimator, grid, cv=StratifiedKFold(3, shuffle=True, random_state=SEED),
                      scoring="accuracy", n_jobs=-1)
    gs.fit(X[tr_idx], y_tr)
    evaluate(name, gs.predict(X), time.time() - t)
    print("   best params:", gs.best_params_)
    return gs

## 4. Spectral-only classifiers (raw 200 bands)
### 4.1 k-Nearest Neighbours

In [ ]:
knn_grid = {"n_neighbors": [1, 3, 5, 7, 9, 11, 15], "weights": ["uniform", "distance"]}
gs_knn = tune_and_run("kNN (200 bands)", KNeighborsClassifier(), knn_grid, X_spec)

# CV accuracy vs k
cvr = pd.DataFrame(gs_knn.cv_results_)
plt.figure(figsize=(5, 3.5))
for w in ["uniform", "distance"]:
    s = cvr[cvr.param_weights == w]
    plt.plot(s.param_n_neighbors.astype(int), 100 * s.mean_test_score, marker="o", label=w)
plt.xlabel("k"); plt.ylabel("3-fold CV accuracy (%)"); plt.title("kNN tuning")
plt.legend(); plt.grid(alpha=.3); plt.tight_layout(); plt.show()

### 4.2 Logistic Regression and LDA (linear baselines)

In [ ]:
t = time.time()
lr = LogisticRegression(C=1.0, max_iter=2000, n_jobs=-1).fit(X_spec[tr_idx], y_tr)
evaluate("Logistic Regression (200 bands)", lr.predict(X_spec), time.time() - t)

t = time.time()
lda = LinearDiscriminantAnalysis().fit(X_spec[tr_idx], y_tr)
evaluate("LDA (200 bands)", lda.predict(X_spec), time.time() - t)

### 4.3 Random Forest

In [ ]:
rf_grid = {"n_estimators": [200, 500], "max_features": ["sqrt", 0.2]}
gs_rf = tune_and_run("Random Forest (200 bands)",
                     RandomForestClassifier(random_state=SEED, n_jobs=-1), rf_grid, X_spec)

# which bands matter?
imp = gs_rf.best_estimator_.feature_importances_
plt.figure(figsize=(8, 3)); plt.bar(range(B), imp, width=1.0)
plt.xlabel("Band index"); plt.ylabel("Importance"); plt.title("Random Forest band importance")
plt.tight_layout(); plt.show()

### 4.4 SVM (RBF kernel)

In [ ]:
svm_grid = {"C": [10, 100, 1000], "gamma": [0.0005, 0.001, 0.005]}
gs_svm = tune_and_run("SVM-RBF (200 bands)", SVC(kernel="rbf"), svm_grid, X_spec)

## 5. PCA dimensionality reduction

PCA is fitted on **training pixels only**, then applied to every pixel.

In [ ]:
pca = PCA(n_components=50, random_state=SEED).fit(X_spec[tr_idx])
cum = np.cumsum(pca.explained_variance_ratio_) * 100

plt.figure(figsize=(5, 3.5))
plt.plot(range(1, 51), cum, marker=".")
plt.axhline(99, ls="--", c="gray")
plt.xlabel("# components")
plt.ylabel("Cumulative variance (%)")
plt.title("PCA explained variance")
plt.grid(alpha=.3)
plt.tight_layout()
plt.show()
print({k: int(np.searchsorted(cum, k) + 1) for k in (95, 99, 99.9)},
      "components for 95 / 99 / 99.9 % variance")

N_PCA = 30
Z_all = pca.transform(X_spec)[:, :N_PCA]
Z_scaler = StandardScaler().fit(Z_all[tr_idx])
Z_all = Z_scaler.transform(Z_all).astype(np.float32)
print("PCA features:", Z_all.shape)

In [ ]:
gs_knn_pca = tune_and_run("kNN (PCA-30)", KNeighborsClassifier(), knn_grid, Z_all)
gs_svm_pca = tune_and_run("SVM-RBF (PCA-30)", SVC(kernel="rbf"),
                          {"C": [10, 100, 1000], "gamma": [0.005, 0.01, 0.05]}, Z_all)
gs_rf_pca  = tune_and_run("Random Forest (PCA-30)",
                          RandomForestClassifier(random_state=SEED, n_jobs=-1),
                          {"n_estimators": [200, 500], "max_features": ["sqrt", 0.3]}, Z_all)

## 6. Spectral-spatial classification (pixel + neighbourhood)

Each pixel is described by the `P x P` window of PCA bands around it. This injects spatial context.

In [ ]:
cube_pca = Z_all.reshape(H, W, N_PCA)      # PCA image (H, W, 30)

def extract_patches(img, idx, P):
    """img: (H, W, C) -> patches (len(idx), P, P, C), reflect-padded at the borders."""
    r = P // 2
    padded = np.pad(img, ((r, r), (r, r), (0, 0)), mode="reflect")
    rows, cols = idx // W, idx % W
    return np.stack([padded[i:i + P, j:j + P, :] for i, j in zip(rows, cols)])

P_CLS, C_CLS = 9, 15          # window size and number of PCA bands for the classical models
all_idx = np.arange(N)
X_patch = extract_patches(cube_pca[:, :, :C_CLS], all_idx, P_CLS).reshape(N, -1)
print("flattened patch features:", X_patch.shape)

In [ ]:
gs_knn_sp = tune_and_run("kNN (PCA + 9x9 patch)", KNeighborsClassifier(), knn_grid, X_patch)
gs_svm_sp = tune_and_run("SVM-RBF (PCA + 9x9 patch)", SVC(kernel="rbf"),
                         {"C": [10, 100, 1000], "gamma": [0.0005, 0.001, 0.005]}, X_patch)
gs_rf_sp  = tune_and_run("Random Forest (PCA + 9x9 patch)",
                         RandomForestClassifier(random_state=SEED, n_jobs=-1),
                         {"n_estimators": [300], "max_features": ["sqrt", 0.1]}, X_patch)

## 7. Deep learning (PyTorch)

* **1D-CNN** - spectral-only, input = the 200-band spectrum
* **HybridSN** - 3D convolutions over (band x height x width) followed by a 2D convolution, input = `15x15x30` PCA patch

A validation set (10 % of the *training* pixels) selects the best epoch, so the test set is never used for tuning.
Use a GPU runtime (Colab: *Runtime -> Change runtime type -> GPU*) if you can; CPU works but is slower.

In [ ]:
import copy
try:
    import torch, torch.nn as nn, torch.nn.functional as F
    TORCH_OK = True
except Exception as e:
    TORCH_OK = False
    print("PyTorch not available -> install it with `pip install torch` to run the deep-learning cells.", e)

if TORCH_OK:
    DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    torch.manual_seed(SEED)
    if DEVICE.type == "cuda":
        torch.cuda.manual_seed_all(SEED)

    print("device:", DEVICE)

    # Validation split is drawn only from the training pixels.
    trn_idx, val_idx = train_test_split(
        tr_idx, test_size=0.10, stratify=y_all[tr_idx], random_state=SEED
    )
    print(f"DL train: {len(trn_idx)} | val: {len(val_idx)} | test: {len(te_idx)}")

In [ ]:
def train_model(model, make_input, epochs=60, lr=1e-3, batch=64, wd=1e-4, name="model"):
    """make_input(idx) -> float tensor for those pixel indices. Trains with Adam, keeps best-val weights."""
    model = model.to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=wd)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    Xtr, ytr_t = make_input(trn_idx), torch.tensor(y_all[trn_idx] - 1)
    Xva, yva_t = make_input(val_idx), torch.tensor(y_all[val_idx] - 1)
    best_acc, best_state, hist = 0, None, []
    for ep in range(1, epochs + 1):
        model.train(); perm = torch.randperm(len(Xtr)); tot = 0
        for i in range(0, len(perm), batch):
            b = perm[i:i + batch]
            xb, yb = Xtr[b].to(DEVICE), ytr_t[b].to(DEVICE)
            opt.zero_grad(); loss = F.cross_entropy(model(xb), yb); loss.backward(); opt.step()
            tot += loss.item() * len(b)
        sched.step()
        model.eval()
        with torch.no_grad():
            pr = torch.cat([model(Xva[i:i + 256].to(DEVICE)).argmax(1).cpu() for i in range(0, len(Xva), 256)])
            va = (pr == yva_t).float().mean().item()
        hist.append((tot / len(Xtr), va))
        if va > best_acc: best_acc, best_state = va, copy.deepcopy(model.state_dict())
        if ep % 10 == 0 or ep == 1:
            print(f"[{name}] epoch {ep:3d}/{epochs}  train-loss {tot/len(Xtr):.4f}  val-acc {va:.4f}")
    model.load_state_dict(best_state)
    print(f"[{name}] best val-acc: {best_acc:.4f}")
    return model, hist

@torch.no_grad()
def predict_all(model, make_input, chunk=2048):
    """Predict class (1..16) for every pixel in the image."""
    model.eval(); out = []
    for i in range(0, N, chunk):
        xb = make_input(np.arange(i, min(i + chunk, N))).to(DEVICE)
        out.append(model(xb).argmax(1).cpu().numpy() + 1)
    return np.concatenate(out)

def plot_history(hist, title):
    h = np.array(hist); fig, ax = plt.subplots(1, 2, figsize=(9, 3))
    ax[0].plot(h[:, 0]); ax[0].set_title(title + " - train loss"); ax[0].set_xlabel("epoch")
    ax[1].plot(h[:, 1]); ax[1].set_title(title + " - validation acc"); ax[1].set_xlabel("epoch")
    plt.tight_layout(); plt.show()

### 7.1 1D-CNN (spectral)

In [ ]:
class CNN1D(nn.Module):
    def __init__(self, n_bands=B, n_classes=NC):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv1d(1, 32, 7, padding=3), nn.BatchNorm1d(32), nn.ReLU(), nn.MaxPool1d(2),
            nn.Conv1d(32, 64, 5, padding=2), nn.BatchNorm1d(64), nn.ReLU(), nn.MaxPool1d(2),
            nn.Conv1d(64, 128, 3, padding=1), nn.BatchNorm1d(128), nn.ReLU(), nn.MaxPool1d(2))
        with torch.no_grad():
            nflat = self.features(torch.zeros(1, 1, n_bands)).numel()
        self.head = nn.Sequential(nn.Flatten(), nn.Linear(nflat, 128), nn.ReLU(),
                                  nn.Dropout(0.4), nn.Linear(128, n_classes))
    def forward(self, x): return self.head(self.features(x))

spec_tensor = torch.tensor(X_spec, dtype=torch.float32)
make_1d = lambda idx: spec_tensor[idx].unsqueeze(1)          # (n, 1, 200)

t = time.time()
cnn1d, hist1d = train_model(CNN1D(), make_1d, epochs=60, lr=1e-3, name="1D-CNN")
evaluate("1D-CNN (200 bands)", predict_all(cnn1d, make_1d), time.time() - t)
plot_history(hist1d, "1D-CNN")

### 7.2 HybridSN (3D-CNN + 2D-CNN on 15x15x30 PCA patches)

In [ ]:
P_DL = 15                                                        # patch size for HybridSN
padded_pca = np.pad(cube_pca, ((P_DL // 2,) * 2, (P_DL // 2,) * 2, (0, 0)), mode="reflect")

def make_patch(idx):
    """-> tensor (n, 1, bands=30, P, P)"""
    rows, cols = idx // W, idx % W
    p = np.stack([padded_pca[i:i + P_DL, j:j + P_DL, :] for i, j in zip(rows, cols)])   # (n,P,P,30)
    return torch.tensor(p.transpose(0, 3, 1, 2), dtype=torch.float32).unsqueeze(1)

class HybridSN(nn.Module):
    def __init__(self, bands=N_PCA, patch=P_DL, n_classes=NC):
        super().__init__()
        self.conv3d = nn.Sequential(
            nn.Conv3d(1, 8, (7, 3, 3)), nn.BatchNorm3d(8), nn.ReLU(),
            nn.Conv3d(8, 16, (5, 3, 3)), nn.BatchNorm3d(16), nn.ReLU(),
            nn.Conv3d(16, 32, (3, 3, 3)), nn.BatchNorm3d(32), nn.ReLU())
        with torch.no_grad():
            s = self.conv3d(torch.zeros(1, 1, bands, patch, patch)).shape      # (1,32,d,h,w)
        c2d = s[1] * s[2]
        self.conv2d = nn.Sequential(nn.Conv2d(c2d, 64, 3), nn.BatchNorm2d(64), nn.ReLU())
        with torch.no_grad():
            nflat = self.conv2d(torch.zeros(1, c2d, s[3], s[4])).numel()
        self.fc = nn.Sequential(nn.Flatten(), nn.Linear(nflat, 256), nn.ReLU(), nn.Dropout(0.4),
                                nn.Linear(256, 128), nn.ReLU(), nn.Dropout(0.4), nn.Linear(128, n_classes))
    def forward(self, x):
        x = self.conv3d(x)
        x = x.reshape(x.shape[0], -1, x.shape[3], x.shape[4])    # merge channel & band dims
        return self.fc(self.conv2d(x))

print(HybridSN())

In [ ]:
t = time.time()
hybrid, hist_h = train_model(HybridSN(), make_patch, epochs=40, lr=1e-3, batch=64, name="HybridSN")
evaluate("HybridSN (PCA-30, 15x15)", predict_all(hybrid, make_patch), time.time() - t)
plot_history(hist_h, "HybridSN")

## 8. Results
### 8.1 Comparison table (single split, test pixels only)

In [ ]:
res = pd.DataFrame(results).T.sort_values("OA", ascending=False)
res["Kappa"] = res["Kappa"].astype(float).round(4)
display(res.round(2))
res.to_csv("results_single_split.csv")

fig, ax = plt.subplots(figsize=(8, 0.42 * len(res) + 1))
ax.barh(res.index[::-1], res["OA"][::-1], color="steelblue")
for i, v in enumerate(res["OA"][::-1]): ax.text(v + 0.2, i, f"{v:.1f}", va="center", fontsize=8)
ax.set_xlim(max(0, res["OA"].min() - 10), 101); ax.set_xlabel("Overall accuracy (%)")
ax.set_title(f"Indian Pines - {int(TRAIN_FRAC*100)}% training pixels"); plt.tight_layout(); plt.show()

### 8.2 Per-class accuracy

In [ ]:
pc = pd.DataFrame(per_class_acc, index=CLASS_NAMES).round(1)
top = res.index[:6]
display(pc[top].style.background_gradient(cmap="RdYlGn", vmin=0, vmax=100))

### 8.3 Best model: confusion matrix and classification maps

In [ ]:
best = res.index[0]
print("Best model:", best)
yt, yp = y_all[te_idx], pred_all[best][te_idx]
cm = confusion_matrix(yt, yp, labels=CLASSES)
cm_n = cm / cm.sum(1, keepdims=True)

plt.figure(figsize=(8, 6.5))
plt.imshow(cm_n, cmap="Blues", vmin=0, vmax=1)
plt.xticks(range(NC), CLASSES); plt.yticks(range(NC), [f"{c} {n}" for c, n in zip(CLASSES, CLASS_NAMES)], fontsize=7)
for i in range(NC):
    for j in range(NC):
        if cm_n[i, j] > 0.05:
            plt.text(j, i, f"{cm_n[i, j]*100:.0f}", ha="center", va="center", fontsize=6,
                     color="white" if cm_n[i, j] > .5 else "black")
plt.xlabel("Predicted"); plt.ylabel("True"); plt.title(f"Normalised confusion matrix (%) - {best}")
plt.colorbar(); plt.tight_layout(); plt.show()

print(classification_report(yt, yp, labels=CLASSES, target_names=CLASS_NAMES, zero_division=0))

In [ ]:
def show_maps(names):
    n = len(names) + 1
    fig, ax = plt.subplots(1, n, figsize=(3.4 * n, 3.6))
    ax[0].imshow(gt, cmap=cmap, vmin=0, vmax=16, interpolation="nearest"); ax[0].set_title("Ground truth")
    for a, nm in zip(ax[1:], names):
        pm = pred_all[nm].reshape(H, W)
        a.imshow(np.where(gt > 0, pm, 0), cmap=cmap, vmin=0, vmax=16, interpolation="nearest")
        a.set_title(f"{nm}\nOA {results[nm]['OA']:.1f}%", fontsize=8)
    for a in ax: a.axis("off")
    plt.tight_layout(); plt.show()

# kNN vs best spectral-only vs best spectral-spatial vs best overall
show_maps(list(dict.fromkeys(["kNN (200 bands)", "SVM-RBF (200 bands)",
                              "SVM-RBF (PCA + 9x9 patch)", best])))

# Full-scene prediction (every pixel, including unlabeled background) by the best model
plt.figure(figsize=(5, 5))
plt.imshow(pred_all[best].reshape(H, W), cmap=cmap, vmin=1, vmax=16, interpolation="nearest")
plt.title(f"Full-scene prediction - {best}"); plt.axis("off"); plt.show()

## 9. Multiple random splits (mean ± std)

A single split is not enough for a paper. This repeats the fast classical pipelines over several seeds
(scaler and PCA are re-fitted on each training set). Add the CNNs yourself in the same loop if you have time/GPU.

In [ ]:
SEEDS = [0, 1, 2, 3, 4]
best_k  = gs_knn_pca.best_params_
best_sv = gs_svm_pca.best_params_
best_rf = gs_rf_pca.best_params_

def one_run(seed):
    tr, te = train_test_split(labeled_idx, train_size=TRAIN_FRAC, stratify=y_all[labeled_idx], random_state=seed)
    sc = StandardScaler().fit(X_all[tr]); Xs = sc.transform(X_all)
    pc_ = PCA(N_PCA, random_state=seed).fit(Xs[tr]); Z = pc_.transform(Xs)
    Z = StandardScaler().fit(Z[tr]).transform(Z)
    models = {
        "kNN (PCA-30)": KNeighborsClassifier(**best_k),
        "SVM-RBF (PCA-30)": SVC(kernel="rbf", **best_sv),
        "Random Forest (PCA-30)": RandomForestClassifier(random_state=seed, n_jobs=-1, **best_rf),
    }
    # spectral-spatial SVM
    cp = Z.reshape(H, W, N_PCA)[:, :, :C_CLS]
    Xp = extract_patches(cp, all_idx, P_CLS).reshape(N, -1)
    out = {}
    for nm, m in models.items():
        m.fit(Z[tr], y_all[tr]); p = m.predict(Z[te])
        out[nm] = (accuracy_score(y_all[te], p) * 100,
                   recall_score(y_all[te], p, labels=CLASSES, average="macro", zero_division=0) * 100,
                   cohen_kappa_score(y_all[te], p))
    m = SVC(kernel="rbf", **gs_svm_sp.best_params_).fit(Xp[tr], y_all[tr]); p = m.predict(Xp[te])
    out["SVM-RBF (PCA + 9x9 patch)"] = (accuracy_score(y_all[te], p) * 100,
                   recall_score(y_all[te], p, labels=CLASSES, average="macro", zero_division=0) * 100,
                   cohen_kappa_score(y_all[te], p))
    return out

runs = [one_run(s) for s in SEEDS]
rows = []
for nm in runs[0]:
    arr = np.array([r[nm] for r in runs])
    rows.append({"Model": nm,
                 "OA": f"{arr[:,0].mean():.2f} ± {arr[:,0].std():.2f}",
                 "AA": f"{arr[:,1].mean():.2f} ± {arr[:,1].std():.2f}",
                 "Kappa": f"{arr[:,2].mean():.4f} ± {arr[:,2].std():.4f}"})
multi = pd.DataFrame(rows).set_index("Model")
display(multi)
multi.to_csv("results_multi_seed.csv")

## 10. Notes for the report

* **Leakage warning:** a random pixel split lets neighbouring pixels (and overlapping patches) fall in both train and test,
  which inflates accuracy, especially for patch-based and CNN models. State this limitation, or add a spatially
  disjoint split (e.g. train on one image region, test on another).
* **Class imbalance:** report OA, AA and Kappa, and discuss the small classes (Alfalfa, Oats, Grass-pasture-mowed have only 20-28 pixels).
* **Expected pattern:** kNN / LR / LDA < RF < SVM on raw bands; PCA mostly keeps accuracy while cutting cost; spatial patches give the biggest jump;
  HybridSN is typically the best, but with a small training set a well-tuned SVM on spectral-spatial features is competitive.
* **Reproducibility:** `SEED`, `TRAIN_FRAC`, `N_PCA`, patch sizes and the CV grids are all defined at the top of each section.